# 03 — Main Orchestrator dispatch

Pure-Python dispatch controller. Things you can poke without spawning a real WIO:

- The cycle summary formatter (MORD-05).
- The worktree lifecycle (`_git_worktree_add` / `_git_worktree_remove`).
- The strict env allowlist (T-4-04) — confirmed by source inspection.
- The claim-delay knob (`HSB_CLAIM_DELAY_MS`).

**Side effect:** the worktree section creates a throwaway git repo under `$HSB_NOTEBOOK_SCRATCH_DIR` (defaults to a tmp dir) and removes it on completion. Nothing touches the real repo.

In [ ]:
import asyncio
import os
import shutil
import subprocess
import tempfile
from pathlib import Path

from _helpers import ensure_src_on_path

ensure_src_on_path()
from hsb.agents import main_orchestrator as mo

## Cycle summary formatter (MORD-05)

Plain string-builder over `DispatchedItem`s. We feed a mix and confirm headers + counts.

In [ ]:
from hsb.contracts.main_orchestrator import DispatchedItem

items = [
    DispatchedItem(
        work_item_id="LIN-1",
        orchestrator_instance="cascade-0",
        claim_status="claimed",
        final_status="completed",
    ),
    DispatchedItem(
        work_item_id="LIN-2",
        orchestrator_instance="parallel-0",
        claim_status="claimed",
        final_status="failed",
    ),
    DispatchedItem(
        work_item_id="LIN-3",
        orchestrator_instance="skipped",
        claim_status="skipped",
        final_status="blocked",
    ),
]
summary = mo._build_cycle_summary("parallel", items)
print(summary)
assert "**Mode:** parallel" in summary
assert "**Dispatched:** 3 tasks" in summary
assert "**Completed:** 1" in summary
assert "**Failed/Blocked:** 2" in summary
assert "**Skipped (claim failed):** 1" in summary

## T-4-04 — strict env allowlist for WIO subprocess

Source inspection: confirm `_run_wio_subprocess` does NOT pass `**os.environ` and that the env dict is the documented 5-key allowlist. We can't easily run the real subprocess from a notebook, but we can prove the structural defense exists.

In [ ]:
src = Path(mo.__file__).read_text()
# Anti-pattern check: any '**os.environ' anywhere is a red flag.
assert "**os.environ" not in src, "T-4-04: os.environ wholesale spread detected"
# Positive check: the documented allowlist keys appear once each.
for key in [
    "PATH",
    "HOME",
    "ANTHROPIC_API_KEY",
    "HSB_WIO_INPUT_FILE",
    "HSB_WIO_OUTPUT_FILE",
]:
    assert f'"{key}"' in src, f"T-4-04: missing allowlist key {key}"
print("T-4-04: subprocess env is the strict 5-key allowlist (no os.environ wholesale)")

## Worktree add / remove (MORD-04 + D-09)

Create a throwaway git repo, run `_git_worktree_add` on it, confirm the path exists, then run `_git_worktree_remove` and confirm it's gone. Cleanup runs in `finally` so a failure doesn't leave junk on disk.

In [ ]:
scratch = Path(
    os.environ.get("HSB_NOTEBOOK_SCRATCH_DIR") or tempfile.mkdtemp(prefix="hsb-nb-")
)
repo = scratch / "repo"
repo.mkdir(parents=True, exist_ok=True)
try:
    # Initialize a git repo with a single commit so worktrees can branch from it.
    subprocess.run(["git", "init", "-q", "-b", "main"], cwd=repo, check=True)
    subprocess.run(
        ["git", "config", "user.email", "nb@example.com"], cwd=repo, check=True
    )
    subprocess.run(["git", "config", "user.name", "nb"], cwd=repo, check=True)
    (repo / "README.md").write_text("seed\n")
    subprocess.run(["git", "add", "."], cwd=repo, check=True)
    subprocess.run(["git", "commit", "-q", "-m", "seed"], cwd=repo, check=True)

    # Add a worktree via the orchestrator's helper.
    wt = asyncio.run(
        mo._git_worktree_add(str(repo), task_id="42", branch_name="feature/LIN-42-x")
    )
    assert Path(wt).exists()
    print("worktree at", wt)

    # Confirm git lists it.
    listed = subprocess.run(
        ["git", "worktree", "list"],
        cwd=repo,
        check=True,
        capture_output=True,
        text=True,
    ).stdout
    assert "LIN-42" in listed, listed

    # Remove (best-effort cleanup). D-09: the function never raises.
    asyncio.run(mo._git_worktree_remove(str(repo), task_id="42"))
    listed = subprocess.run(
        ["git", "worktree", "list"],
        cwd=repo,
        check=True,
        capture_output=True,
        text=True,
    ).stdout
    assert "LIN-42" not in listed, listed
    print("worktree removed cleanly")
finally:
    shutil.rmtree(scratch, ignore_errors=True)

## Inter-claim delay (D-06)

`HSB_CLAIM_DELAY_MS` is read once at import; default 200ms. Confirm the constant is exposed and overridable.

In [ ]:
import importlib

# Default
os.environ.pop("HSB_CLAIM_DELAY_MS", None)
importlib.reload(mo)
assert mo.CLAIM_DELAY_MS == 200, mo.CLAIM_DELAY_MS
# Override
os.environ["HSB_CLAIM_DELAY_MS"] = "500"
importlib.reload(mo)
assert mo.CLAIM_DELAY_MS == 500
del os.environ["HSB_CLAIM_DELAY_MS"]
importlib.reload(mo)
print("CLAIM_DELAY_MS knob: env-overridable, defaults to 200ms")

## Pitfall C — stale-worktree prune at parallel-mode startup

Source inspection: `_parallel_dispatch` runs `git worktree prune` before adding any new ones. Cheap but important — without this, a crashed prior run leaves orphan refs that block a new add.

In [ ]:
src = Path(mo.__file__).read_text()
# git worktree prune appears at the top of _parallel_dispatch.
assert "git worktree prune" in src or '"prune"' in src, (
    "parallel dispatch lacks worktree prune"
)
print("Pitfall C: parallel dispatch prunes stale worktrees at startup")